# 3. Phân Tích Khám Phá Dữ Liệu (EDA)

Notebook này thực hiện phân tích khám phá trên dữ liệu Olist đã qua ETL. Mục tiêu:
- Hiểu phân bố dữ liệu và xu hướng kinh doanh
- Phát hiện các mẫu hành vi khách hàng
- Tìm insight để hỗ trợ ra quyết định
- Chuẩn bị cho việc xây dựng mô hình ML

> 📌 Notebook này tương ứng với file `spark_jobs/eda.py` trong project.

## 3.1 Import thư viện và đọc dữ liệu

In [ ]:
import os
os.environ["PYSPARK_PYTHON"] = "C:/Users/Admin/AppData/Local/Programs/Python/Python313/python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = os.environ["PYSPARK_PYTHON"]
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["TZ"] = "UTC"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Olist_EDA_Notebook")
    .master("local[4]")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Đọc dữ liệu từ HDFS Silver
HDFS_SILVER = "hdfs://localhost:9000/user/bigdata/olist/silver"
merged = spark.read.parquet(f"{HDFS_SILVER}/merged_orders")
print(f"✅ Đã đọc merged_orders: {merged.count():,} dòng, {len(merged.columns)} cột")
merged.printSchema()

## 3.2 Thống kê tổng quan (Overview Statistics)

Đầu tiên, hãy xem tổng quan các chỉ số kinh doanh chính. Đây là những con số mà doanh nghiệp cần biết đầu tiên.

In [ ]:
# Tổng doanh thu
total_revenue = merged.agg(F.sum("order_value")).collect()[0][0]
# Tổng đơn hàng
total_orders = merged.count()
# Đánh giá trung bình
avg_review = merged.agg(F.avg("review_score")).collect()[0][0]
# Tổng khách hàng
total_customers = merged.select("customer_unique_id").distinct().count()
# Tỷ lệ churn
churn_count = merged.filter(F.col("churn") == 1).count()
churn_rate = churn_count / total_orders * 100

print("=" * 50)
print("📊 THỐNG KÊ TỔNG QUAN")
print("=" * 50)
print(f"  💰 Tổng doanh thu:    R$ {total_revenue:,.2f}")
print(f"  📦 Tổng đơn hàng:    {total_orders:,}")
print(f"  ⭐ Đánh giá TB:      {avg_review:.2f}/5")
print(f"  👥 Tổng khách hàng:  {total_customers:,}")
print(f"  📉 Tỷ lệ Churn:     {churn_rate:.1f}%")

## 3.3 Phân tích Doanh thu theo thời gian

### Tại sao phân tích theo thời gian?
Xu hướng doanh thu giúp doanh nghiệp:
- Nhận biết mùa cao điểm / thấp điểm
- Phát hiện tăng trưởng hay suy thoái
- Lập kế hoạch kinh doanh cho quý/năm tiếp theo

In [ ]:
# Doanh thu theo tháng
monthly_revenue = (
    merged
    .groupBy("purchase_year_month")
    .agg(
        F.sum("order_value").alias("total_revenue"),
        F.count("order_id").alias("order_count"),
    )
    .orderBy("purchase_year_month")
)

print("📊 DOANH THU THEO THÁNG:")
monthly_revenue.show(30, truncate=False)

## 3.4 Phân tích Top Danh mục sản phẩm

Xem danh mục nào bán chạy nhất, doanh thu cao nhất, đánh giá tốt nhất.

In [ ]:
# Thống kê theo danh mục sản phẩm
category_stats = (
    merged
    .groupBy("main_category_english")
    .agg(
        F.sum("order_value").alias("total_revenue"),
        F.count("order_id").alias("order_count"),
        F.avg("total_price").alias("avg_price"),
        F.avg("review_score").alias("avg_review"),
    )
    .orderBy(F.col("total_revenue").desc())
)

print("📊 TOP 10 DANH MỤC THEO DOANH THU:")
category_stats.show(10, truncate=False)

## 3.5 Phân tích Phân khúc khách hàng (Customer Segmentation)

Dựa trên RFM Score, khách hàng được phân thành các nhóm:
- **Champions** (rfm_score >= 10): Khách VIP, mua nhiều, chi nhiều
- **Loyal** (8-9): Khách trung thành
- **Potential** (6-7): Có tiềm năng phát triển
- **At Risk** (4-5): Có nguy cơ rời bỏ
- **Lost** (3): Đã mất

In [ ]:
# Phân khúc khách hàng dựa trên RFM Score
rfm_data = merged.select("customer_unique_id", "rfm_score").distinct().filter(F.col("rfm_score").isNotNull())

rfm_data = rfm_data.withColumn(
    "segment",
    F.when(F.col("rfm_score") >= 10, "Champions")
     .when(F.col("rfm_score") >= 8, "Loyal")
     .when(F.col("rfm_score") >= 6, "Potential")
     .when(F.col("rfm_score") >= 4, "At Risk")
     .otherwise("Lost")
)

print("📊 PHÂN KHÚC KHÁCH HÀNG:")
segment_counts = rfm_data.groupBy("segment").count().orderBy(F.col("count").desc())
segment_counts.show()

total_cust = rfm_data.count()
for row in segment_counts.collect():
    pct = row["count"] / total_cust * 100
    print(f"  {row['segment']}: {row['count']:,} ({pct:.1f}%)")

## 3.6 Phân tích Churn (Rời bỏ khách hàng)

### Câu hỏi cần trả lời:
- Tỷ lệ churn là bao nhiêu?
- Nhóm khách hàng nào có tỷ lệ churn cao nhất?
- Yếu tố nào ảnh hưởng đến churn?

In [ ]:
# Phân bố Churn
print("📊 PHÂN BỐ CHURN:")
merged.groupBy("churn").count().show()

# Churn theo segment (nếu có)
print("📊 TỶ LỆ CHURN THEO PHÂN KHÚC:")
churn_by_segment = (
    merged
    .filter(F.col("rfm_score").isNotNull())
    .withColumn("segment",
        F.when(F.col("rfm_score") >= 10, "Champions")
         .when(F.col("rfm_score") >= 8, "Loyal")
         .when(F.col("rfm_score") >= 6, "Potential")
         .when(F.col("rfm_score") >= 4, "At Risk")
         .otherwise("Lost")
    )
    .groupBy("segment")
    .agg(
        F.count("*").alias("total"),
        F.sum("churn").alias("churned"),
    )
    .withColumn("churn_rate", F.round(F.col("churned") / F.col("total") * 100, 1))
    .orderBy("churn_rate")
)
churn_by_segment.show()

## 3.7 Phân tích Logistics

### Thời gian giao hàng ảnh hưởng thế nào đến đánh giá?
Giả thuyết: Giao hàng càng nhanh → đánh giá càng cao.

In [ ]:
# Thống kê thời gian giao hàng
print("📊 THỐNG KÊ THỜI GIAN GIAO HÀNG:")
delivered = merged.filter(F.col("delivery_days").isNotNull())
delivered.select("delivery_days").describe().show()

# Tỷ lệ giao hàng đúng hạn vs trễ
print("\n📊 TRẠNG THÁI GIAO HÀNG:")
merged.groupBy("delivery_status").count().orderBy(F.col("count").desc()).show()

# Ảnh hưởng của thời gian giao hàng đến đánh giá
print("\n📊 ĐÁNH GIÁ TRUNG BÌNH THEO TRẠNG THÁI GIAO HÀNG:")
merged.filter(F.col("delivery_status") != "not_delivered").groupBy("delivery_status").agg(
    F.avg("review_score").alias("avg_review"),
    F.count("*").alias("count")
).show()

## 3.8 Phân tích Địa lý

Xem bang nào đóng góp doanh thu nhiều nhất.

In [ ]:
# Doanh thu theo bang
print("📊 TOP 10 BANG THEO DOANH THU:")
state_revenue = (
    merged
    .groupBy("customer_state")
    .agg(
        F.sum("order_value").alias("revenue"),
        F.count("*").alias("orders"),
    )
    .orderBy(F.col("revenue").desc())
)
state_revenue.show(10)

## 3.9 Phân tích Thanh toán

Xem phương thức thanh toán nào được ưa chuộng nhất.

In [ ]:
# Phân bố phương thức thanh toán
print("📊 PHƯƠNG THỨC THANH TOÁN:")
payment_dist = (
    merged
    .groupBy("payment_type")
    .agg(F.count("*").alias("count"), F.sum("total_payment_value").alias("total_value"))
    .orderBy(F.col("count").desc())
)
payment_dist.show()

## 3.10 Kết luận từ EDA

### Insights chính:
1. **Doanh thu** tăng trưởng mạnh từ 2017 đến giữa 2018
2. **Sức khoẻ & Làm đẹp** là danh mục bán chạy nhất
3. **São Paulo (SP)** đóng góp doanh thu lớn nhất
4. **Credit card** là phương thức thanh toán phổ biến nhất
5. **Thời gian giao hàng** có tương quan nghịch với đánh giá (giao trễ → đánh giá thấp)
6. **Tỷ lệ churn cao (~60%)** - cần xây dựng mô hình dự đoán

**Bước tiếp theo** → Notebook `04_ml_models.ipynb`: Xây dựng mô hình Machine Learning.

In [ ]:
spark.stop()
print('✅ SparkSession đã dừng.')